# Day 3 · 실습 02. ResNet18 전이학습 전략 비교

**연계 이론:** 이론 1(Pre-training과 Transfer Learning), 이론 2(전이학습의 학습 전략)

사전학습된 ResNet18의 Backbone 전체 동결과 `layer4` 동결 해제를 비교합니다.
학습 가능한 파라미터 수와 Training/Validation Loss의 차이를 확인합니다.

데이터는 `assets/`, 모델 가중치는 `weights/`, 실습 결과는 `outputs/`에 저장합니다.
`DAY3_PROJECT_ROOT`를 지정하면 전체 실습의 기준 폴더를 바꿀 수 있습니다.

In [ ]:
import os
import random
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path(os.getenv("DAY3_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
ASSETS_DIR = PROJECT_ROOT / "assets"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_ROOT = Path(os.getenv("DAY3_DATA_ROOT", str(ASSETS_DIR / "hymenoptera_data"))).expanduser().resolve()
for folder in (ASSETS_DIR, WEIGHTS_DIR, OUTPUTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(WEIGHTS_DIR / "torch_hub"))
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "없음")
print("데이터:", DATA_ROOT)

## 1. 개미·벌 데이터셋 준비

공식 PyTorch 튜토리얼의 `hymenoptera_data`를 사용합니다.
데이터가 없을 때만 다운로드하며, 세 실습이 같은 경로를 재사용합니다.

In [ ]:
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

def ensure_dataset(root=DATA_ROOT):
    required = [root / split / name for split in ("train", "val") for name in ("ants", "bees")]
    if all(p.is_dir() for p in required):
        print("기존 데이터 사용:", root)
        return root

    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "hymenoptera_data.zip"
    if not archive.exists():
        print("개미·벌 데이터 다운로드 중...")
        temp = archive.with_suffix(".zip.part")
        try:
            with urllib.request.urlopen(DATA_URL, timeout=90) as response, temp.open("wb") as out:
                shutil.copyfileobj(response, out)
            temp.replace(archive)
        finally:
            temp.unlink(missing_ok=True)

    # ZIP 내부의 최상위 폴더 이름과 관계없이 DATA_ROOT에 정확히 풉니다.
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            parts = Path(member.filename).parts
            if not parts or parts[0] != "hymenoptera_data":
                raise ValueError("예상하지 못한 ZIP 경로: " + member.filename)
            if len(parts) == 1:
                continue
            destination = (root / Path(*parts[1:])).resolve()
            if not destination.is_relative_to(root.resolve()):
                raise ValueError("허용되지 않은 ZIP 경로: " + member.filename)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst)

    if not all(p.is_dir() for p in required):
        raise FileNotFoundError(f"train/val/ants/bees 폴더를 확인하세요: {root}")
    print("데이터 준비 완료:", root)
    return root

ensure_dataset()

## 2. 학습 데이터와 실험 조건

두 전략의 비교가 가능하도록 동일한 이미지, 전처리, 학습률, Epoch를 사용합니다.
무작위 증강은 시각화만 하며 학습 입력에는 적용하지 않습니다.

In [ ]:
TRAIN_PER_CLASS = 32
VAL_PER_CLASS = 12
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 3e-4

base_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
full_train = datasets.ImageFolder(DATA_ROOT / "train", transform=base_transform)
full_val = datasets.ImageFolder(DATA_ROOT / "val", transform=base_transform)
assert full_train.classes == full_val.classes == ["ants", "bees"]
CLASS_NAMES = full_train.classes

def stratified_indices(dataset, per_class, seed):
    rng = np.random.default_rng(seed)
    selected = []
    for label in range(len(dataset.classes)):
        candidates = np.flatnonzero(np.asarray(dataset.targets) == label)
        if len(candidates) < per_class:
            raise ValueError(f"클래스 {label}의 이미지가 부족합니다.")
        selected.extend(rng.choice(candidates, size=per_class, replace=False).tolist())
    return sorted(selected)

train_set = Subset(full_train, stratified_indices(full_train, TRAIN_PER_CLASS, SEED))
val_set = Subset(full_val, stratified_indices(full_val, VAL_PER_CLASS, SEED))
print("Classes:", CLASS_NAMES)
print("Train:", len(train_set), "| Validation:", len(val_set))
print("Input Shape:", tuple(train_set[0][0].shape))

## 3. 데이터 증강 결과 관찰

한 장의 이미지에 Flip, Crop, Rotation, Color Jitter를 각각 적용합니다.
이 결과는 증강의 효과를 눈으로 확인하기 위한 것으로, 아래 학습에는 사용하지 않습니다.

In [ ]:
example_path, _ = full_train.samples[0]
with Image.open(example_path) as im:
    original = im.convert("RGB")

augments = {
    "Original": lambda img: img,
    "Horizontal Flip": transforms.RandomHorizontalFlip(p=1.0),
    "Random Crop": transforms.RandomResizedCrop((224, 224), scale=(0.65, 0.85)),
    "Rotation": transforms.RandomRotation((25, 25)),
    "Color Jitter": transforms.ColorJitter(brightness=0.65, contrast=0.5),
}
fig, axes = plt.subplots(1, len(augments), figsize=(15, 3.5))
for ax, (name, operation) in zip(axes, augments.items()):
    ax.imshow(operation(original.copy()))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. ResNet18의 동결 범위 비교

실험 A: Backbone 전체 동결 + 새로운 Head 학습
실험 B: `layer4` 동결 해제 + 새로운 Head 학습
두 실험은 같은 사전학습 가중치와 같은 초기 Head로 시작합니다.

In [ ]:
WEIGHTS = ResNet18_Weights.IMAGENET1K_V1

def create_model(strategy):
    if strategy not in ("feature_extraction", "partial_finetuning"):
        raise ValueError("지원하지 않는 전략입니다: " + strategy)
    torch.manual_seed(SEED)
    model = models.resnet18(weights=WEIGHTS)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    if strategy == "partial_finetuning":
        for param in model.layer4.parameters():
            param.requires_grad = True
    return model.to(DEVICE)

def trainable_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for strategy in ("feature_extraction", "partial_finetuning"):
    model = create_model(strategy)
    total = sum(p.numel() for p in model.parameters())
    names = [name for name, p in model.named_parameters() if p.requires_grad]
    print(f"{strategy}: {trainable_count(model):,} / {total:,} trainable")
    print("학습 대상 예:", names[:3], "...", names[-2:])
    del model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 5. 두 전략 학습 및 Loss 비교

`requires_grad`로 학습 범위를 설정하고 AdamW로 가중치를 업데이트합니다.
BatchNorm의 Running Mean/Variance는 두 조건 모두 고정합니다.

In [ ]:
def make_loaders():
    generator = torch.Generator().manual_seed(SEED)
    return {
        "train": DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                            generator=generator, num_workers=0,
                            pin_memory=(DEVICE.type == "cuda")),
        "val": DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type == "cuda")),
    }

def one_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()  # BN의 이동 통계는 고정, 학습 가능한 가중치는 그대로 유지

    total_loss, total_correct, total_seen = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_seen += images.size(0)
    return total_loss / total_seen, total_correct / total_seen

def run_experiment(strategy):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    model = create_model(strategy)
    loaders = make_loaders()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE
    )
    history = []
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = one_epoch(model, loaders["train"], criterion, optimizer)
        val_loss, val_acc = one_epoch(model, loaders["val"], criterion)
        history.append({
            "strategy": strategy, "epoch": epoch,
            "train_loss": train_loss, "val_loss": val_loss,
            "train_acc": train_acc, "val_acc": val_acc,
        })
        print(f"{strategy} | epoch {epoch}/{EPOCHS} | "
              f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.3f}")
    result = {
        "strategy": strategy,
        "trainable_params": trainable_count(model),
        "elapsed_sec": time.perf_counter() - start,
        "history": history,
    }
    del model, optimizer
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result

results = [run_experiment("feature_extraction"),
           run_experiment("partial_finetuning")]

## 6. 결과 표·그래프와 저장

학습 가능 파라미터 수, Training/Validation Loss, 검증 정확도를 비교합니다.
짧은 실험 결과만으로 어느 전략이 항상 우수하다고 판단하지 않습니다.

In [ ]:
summary = pd.DataFrame([{
    "strategy": r["strategy"],
    "trainable_params": r["trainable_params"],
    "elapsed_sec": round(r["elapsed_sec"], 2),
    "final_train_loss": round(r["history"][-1]["train_loss"], 4),
    "final_val_loss": round(r["history"][-1]["val_loss"], 4),
    "final_val_acc": round(r["history"][-1]["val_acc"], 3),
} for r in results])
print(summary.to_string(index=False))

history_df = pd.DataFrame([row for r in results for row in r["history"]])
fig, ax = plt.subplots(figsize=(8, 4.5))
for strategy in history_df["strategy"].unique():
    frame = history_df[history_df["strategy"] == strategy]
    ax.plot(frame["epoch"], frame["train_loss"], marker="o", label=f"{strategy} / train")
    ax.plot(frame["epoch"], frame["val_loss"], marker="s", linestyle="--",
            label=f"{strategy} / val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Feature Extraction vs Partial Fine-tuning")
ax.set_xticks(list(range(1, EPOCHS + 1)))
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

summary.to_csv(OUTPUTS_DIR / "day3_02_strategy_summary.csv", index=False, encoding="utf-8-sig")
history_df.to_csv(OUTPUTS_DIR / "day3_02_loss_history.csv", index=False, encoding="utf-8-sig")
print("결과 저장:", OUTPUTS_DIR)

## 7. 이번 실습 간단하게 요약

- Backbone 동결 범위에 따라 학습 가능한 파라미터 수가 달라집니다.
- 두 전략을 동일한 조건에서 학습하고 Loss·Accuracy를 비교했습니다.
- 데이터 증강은 변화만 관찰했으며 두 전략의 학습 조건에는 사용하지 않았습니다.
- 다음 실습에서는 DINOv3로 이미지를 Embedding으로 표현하고 유사도를 비교합니다.